# Module 3: Data Cleaning
**Real-Time Payment Fraud Detection for an E-Commerce Platform**

All cleaning decisions below follow the project announcement (`Ecommerce_Fraud_Detection_Project_Announcement.pdf`) and team requirements:
1. **Completeness, Duplicates & Category Consistency (`[Q1.1]`)**: Verify `0` null cells, `0` duplicate rows, `0` negative amounts/balances, valid sender prefix `C`, and destination prefixes `C`/`M` (`M` = merchant accounts in `PAYMENT`). Ensure monotonic `step` order across `[1, 743]`.
2. **Full Transaction Stream Retention (All 5 Types Retained)**: Per team decision, **all 5 transaction types (`CASH_IN`, `CASH_OUT`, `DEBIT`, `PAYMENT`, `TRANSFER` — all `6,362,620` rows) are retained** in the cleaned dataset rather than dropping `CASH_IN`, `DEBIT`, and `PAYMENT`. Keeping all 5 types preserves complete customer behavioral and account velocity history across the full e-commerce payment stream.
3. **Outliers & Simulator Artefacts Treatment (`[Q1.3]`)**:
   * **High-value outliers ($> Q_3 + 3 \times IQR$ per transaction type)**: Computed separately for each of the 5 types (`CASH_IN`, `CASH_OUT`, `DEBIT`, `PAYMENT`, `TRANSFER`). In `CASH_OUT` and `TRANSFER`, fraud rates inside outliers (`43.56%` and `2.84%`) are far higher than outside (`0.11%` and `0.69%`). Outliers are critical risk indicators and are flagged via `is_amount_outlier_3xIQR` rather than deleted.
   * **Zero-amount transactions (`amount == 0`)**: Exactly `16` rows in the log have `amount == 0`, and all `16` are labeled `isFraud == 1` (simulator artefact in `CASH_OUT`). Flagged explicitly via `is_zero_amount` rather than dropped.
4. **Leakage Exclusion (`[Q1.4]`)**: Dataset authors state cancelled fraud transactions leave balances inconsistent (`oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, `newbalanceDest`), and merchant destinations (`M...` in `PAYMENT`) have structural zero balances. These four balance columns and the baseline rule column (`isFlaggedFraud`) are strictly excluded from predictive features.

In [1]:
# ===== [M3.0] Setup & Load Augmented Dataset from Module 1 (All 5 Transaction Types) =====
import os, warnings
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
np.random.seed(42)

DATA_PATH = r"C:\LONG\Project\BA Project\PS_20174392719_1491204439457_log.csv"
OUT_DIR = r"C:\LONG\Project\BA Project\outputs"
os.makedirs(f"{OUT_DIR}/tables", exist_ok=True)
AUGMENTED_PATH = f"{OUT_DIR}/paysim_augmented_module1.parquet"

TYPES = ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"]
LEAKAGE_COLS = ["oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]
RULE_COL = ["isFlaggedFraud"]

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)

if os.path.exists(AUGMENTED_PATH):
    df_raw = pd.read_parquet(AUGMENTED_PATH)
else:
    # Load raw PaySim and run the exact Module 1 synthetic generator
    df_raw = pd.read_csv(DATA_PATH)
    df_raw["type"] = df_raw["type"].astype(str).str.strip().str.upper().str.replace("-", "_", regex=False)
    if not df_raw["step"].is_monotonic_increasing:
        df_raw = df_raw.sort_values("step", kind="stable").reset_index(drop=True)
    rng = np.random.default_rng(42)
    n_rows = len(df_raw)
    is_fraud = df_raw["isFraud"].to_numpy().astype(bool)
    n_fraud, n_legit = int(is_fraud.sum()), n_rows - int(is_fraud.sum())
    
    age_legit = rng.gamma(3.0, 150.0, size=n_legit)
    young_mask = rng.random(n_fraud) < 0.70
    age_fraud = np.empty(n_fraud)
    age_fraud[young_mask] = rng.exponential(25.0, size=int(young_mask.sum()))
    age_fraud[~young_mask] = rng.gamma(3.0, 150.0, size=int((~young_mask).sum()))
    acc_age = np.empty(n_rows, dtype=np.int32)
    acc_age[~is_fraud] = np.clip(np.rint(age_legit), 1, 3650)
    acc_age[is_fraud] = np.clip(np.rint(age_fraud), 1, 3650)
    df_raw["account_age_days"] = acc_age
    
    trust = np.empty(n_rows, dtype=np.float32)
    trust[~is_fraud] = rng.beta(8.0, 2.0, size=n_legit)
    trust[is_fraud] = rng.beta(2.0, 6.0, size=n_fraud)
    df_raw["device_trust_score"] = np.clip(trust, 0.0, 1.0)
    
    new_dev = np.empty(n_rows, dtype=np.int8)
    new_dev[~is_fraud] = (rng.random(n_legit) < 0.08).astype(np.int8)
    new_dev[is_fraud] = (rng.random(n_fraud) < 0.72).astype(np.int8)
    df_raw["is_new_device"] = new_dev
    
    dist_legit = rng.lognormal(2.5, 1.2, size=n_legit)
    out_m = rng.random(n_legit) < 0.03
    dist_legit[out_m] = rng.uniform(500.0, 5000.0, size=int(out_m.sum()))
    dist_fraud = rng.lognormal(6.2, 1.5, size=n_fraud)
    dist = np.empty(n_rows, dtype=np.float32)
    dist[~is_fraud] = np.clip(dist_legit, 0.0, 15000.0)
    dist[is_fraud] = np.clip(dist_fraud, 0.0, 15000.0)
    df_raw["ip_billing_distance_km"] = dist
    
    mismatch = np.empty(n_rows, dtype=np.int8)
    mismatch[~is_fraud] = (rng.random(n_legit) < 0.05).astype(np.int8)
    mismatch[is_fraud] = (rng.random(n_fraud) < 0.60).astype(np.int8)
    df_raw["shipping_billing_mismatch"] = mismatch
    
    auth = np.empty(n_rows, dtype=np.int8)
    auth[~is_fraud] = np.clip(rng.poisson(0.15, size=n_legit), 0, 15)
    auth[is_fraud] = np.clip(rng.poisson(2.80, size=n_fraud), 0, 15)
    df_raw["failed_auth_attempts_24h"] = auth

In [2]:
# ===== [M3.1] Step-by-Step Cleaning Pipeline with Before/After Documentation (All 5 Types Retained) =====
def audit_snapshot(d: pd.DataFrame, stage: str) -> dict:
    return {
        "stage": stage,
        "rows": len(d),
        "transaction_types": int(d["type"].nunique()),
        "columns": d.shape[1],
        "null_cells": int(d.isna().sum().sum()),
        "duplicate_rows": int(d.duplicated().sum()),
        "fraud_cases": int(d["isFraud"].sum()),
        "fraud_rate_pct": float(100 * d["isFraud"].mean()),
        "total_payment_value": float(d["amount"].sum()),
        "fraud_value": float((d["amount"] * d["isFraud"]).sum()),
        "memory_mb": float(d.memory_usage(deep=True).sum() / (1024 ** 2))
    }

cleaning_log = [audit_snapshot(df_raw, "1. Before Cleaning (All 5 Transaction Types + Balance Cols)")]

# Step 1: Verify string standardization, duplicates, and missing values
df_raw["type"] = df_raw["type"].astype(str).str.strip().str.upper().str.replace("-", "_", regex=False)
df_raw["nameOrig"] = df_raw["nameOrig"].astype(str).str.strip()
df_raw["nameDest"] = df_raw["nameDest"].astype(str).str.strip()
assert df_raw["type"].isin(TYPES).all(), "Invalid transaction type found"
assert int(df_raw.isna().sum().sum()) == 0, "Unexpected null values found"
assert int(df_raw.duplicated().sum()) == 0, "Unexpected duplicate rows found"

if not df_raw["step"].is_monotonic_increasing:
    df_raw = df_raw.sort_values("step", kind="stable").reset_index(drop=True)

# Step 2: Compute 3xIQR outlier fences per transaction type across all 5 types (CASH_IN, CASH_OUT, DEBIT, PAYMENT, TRANSFER)
g_amt = df_raw.groupby("type", observed=True)["amount"]
q1, q3 = g_amt.quantile(0.25), g_amt.quantile(0.75)
upper_fence_3xIQR = (q3 + 3.0 * (q3 - q1)).to_dict()
df_raw["is_amount_outlier_3xIQR"] = (df_raw["amount"] > df_raw["type"].map(upper_fence_3xIQR)).astype("int8")
df_raw["is_zero_amount"] = (df_raw["amount"] == 0).astype("int8")
cleaning_log.append(audit_snapshot(df_raw, "2. After Standardization & Outlier/Artefact Flagging (All 5 Types Kept)"))

# Audit breakdown across all 5 transaction types
type_rows = []
for t in TYPES:
    sub = df_raw[df_raw["type"] == t]
    sub_out = sub[sub["is_amount_outlier_3xIQR"] == 1]
    sub_in = sub[sub["is_amount_outlier_3xIQR"] == 0]
    type_rows.append({
        "type": t,
        "rows_retained": len(sub),
        "share_of_rows_pct": 100.0 * len(sub) / len(df_raw),
        "total_value": float(sub["amount"].sum()),
        "fraud_cases": int(sub["isFraud"].sum()),
        "fraud_rate_pct": float(100.0 * sub["isFraud"].mean()),
        "q1_amount": float(q1[t]),
        "q3_amount": float(q3[t]),
        "upper_fence_3xIQR": float(upper_fence_3xIQR[t]),
        "outliers_3xIQR": len(sub_out),
        "outlier_rate_pct": 100.0 * len(sub_out) / len(sub),
        "fraud_rate_in_outliers_pct": float(100.0 * sub_out["isFraud"].mean()) if len(sub_out) > 0 else 0.0,
        "fraud_rate_non_outliers_pct": float(100.0 * sub_in["isFraud"].mean()) if len(sub_in) > 0 else 0.0,
        "zero_amount_rows": int(sub["is_zero_amount"].sum())
    })
type_audit_df = pd.DataFrame(type_rows).set_index("type")
type_audit_df.to_csv(f"{OUT_DIR}/tables/M3_2_transaction_type_and_outlier_audit.csv")
display(type_audit_df)

# Step 3: Retain ALL 5 transaction types (CASH_IN, CASH_OUT, DEBIT, PAYMENT, TRANSFER) & drop post-decision balance leakage + baseline rule columns
df_clean = df_raw.drop(columns=[c for c in (LEAKAGE_COLS + RULE_COL) if c in df_raw.columns]).copy()
cleaning_log.append(audit_snapshot(df_clean, "3. After Removing Balance Leakage & Baseline Rule Columns (All 5 Types Kept)"))

# Step 4: Optimize data types
df_clean["step"] = df_clean["step"].astype("int32")
df_clean["type"] = pd.Categorical(df_clean["type"], categories=TYPES)
df_clean["isFraud"] = df_clean["isFraud"].astype("int8")
cleaning_log.append(audit_snapshot(df_clean, "4. Final Cleaned Dataset (All 5 Types Kept, Optimized Dtypes)"))

before_after_df = pd.DataFrame(cleaning_log).set_index("stage")
before_after_df.to_csv(f"{OUT_DIR}/tables/M3_1_before_after_cleaning_comparison.csv")
display(before_after_df)

# Save single unified dataset (Synthetic + Cleaned) ready for modeling
clean_path = f"{OUT_DIR}/paysim_final_cleaned.csv"
df_clean.to_csv(clean_path, index=False)
print(f"Saved single final cleaned dataset ({len(df_clean):,} rows x {df_clean.shape[1]} cols, all 5 transaction types) to: {clean_path}")

,rows_retained,share_of_rows_pct,total_value,fraud_cases,fraud_rate_pct,q1_amount,q3_amount,upper_fence_3xIQR,outliers_3xIQR,outlier_rate_pct,fraud_rate_in_outliers_pct,fraud_rate_non_outliers_pct,zero_amount_rows
type,,,,,,,,,,,,,
CASH_IN,"1,399,284",21.99%,"236,367,391,912.46",0,0.00%,"70,510.18","239,899.09","748,065.80","1,097",0.08%,0.00%,0.00%,0
CASH_OUT,"2,237,500",35.17%,"394,412,995,224.49","4,116",0.18%,"72,669.65","246,539.48","768,148.96","3,609",0.16%,43.56%,0.11%,16
DEBIT,"41,432",0.65%,"227,199,221.28",0,0.00%,"1,500.18","5,479.18","17,416.16","1,645",3.97%,0.00%,0.00%,0
PAYMENT,"2,151,495",33.81%,"28,093,371,138.37",0,0.00%,"4,383.82","17,561.22","57,093.42","25,692",1.19%,0.00%,0.00%,0
TRANSFER,"532,909",8.38%,"485,291,987,263.17","4,097",0.77%,"215,905.35","974,958.00","3,252,115.95","19,488",3.66%,2.84%,0.69%,0


,rows,transaction_types,columns,null_cells,duplicate_rows,fraud_cases,fraud_rate_pct,total_payment_value,fraud_value,memory_mb
stage,,,,,,,,,,
1. Before Cleaning (All 5 Transaction Types + Balance Cols),"6,362,620",5,17,0,0,"8,213",0.13%,"1,144,392,944,759.77","12,056,415,427.84",797.24
2. After Standardization & Outlier/Artefact Flagging (All 5 Types Kept),"6,362,620",5,19,0,0,"8,213",0.13%,"1,144,392,944,759.77","12,056,415,427.84",811.65
3. After Removing Balance Leakage & Baseline Rule Columns (All 5 Types Kept),"6,362,620",5,14,0,0,"8,213",0.13%,"1,144,392,944,759.77","12,056,415,427.84",568.93
"4. Final Cleaned Dataset (All 5 Types Kept, Optimized Dtypes)","6,362,620",5,14,0,0,"8,213",0.13%,"1,144,392,944,759.77","12,056,415,427.84",413.91


Saved single final cleaned dataset (6,362,620 rows x 14 cols, all 5 transaction types) to: C:\LONG\Project\BA Project\outputs/paysim_final_cleaned.csv
